# Cross-Domain Celebrity Retrieval — CLIP and ArcFace Fine Tuning with data augmentation


In [1]:
# in order to import functions

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
import clip
from torch.utils.data import DataLoader
print('All imports OK')

# the remaining needed libraries are imported using src files, to avoid clutter

All imports OK


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device : cuda
GPU    : Tesla V100-PCIE-16GB
VRAM   : 16.9 GB


### Data & saving checkpoints

In [3]:
# windows syntax:
# comp_train_dir="..\\data\\competition\\train\\train"
# comp_test_dir="..\\data\\competition\\test"

# linux syntax:
comp_train_dir="../data/competition/train/train"
comp_test_dir="../data/competition/test"

imgs_dir=comp_train_dir

In [4]:
# windows syntax:
# SAVE_DIR = ".\\models\\checkpoints\\aug_clip_arcface_best"

# linux syntax: 
SAVE_DIR = "./models/checkpoints/augm_clip_arcface_best"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {SAVE_DIR}')

Checkpoints will be saved to: ./models/checkpoints/augm_clip_arcface_best


## Cell 5 — Load CLIP ViT-L/14
Downloads ~900 MB weights on first run, cached after that.

In [5]:
print('Loading CLIP ViT-L/14...')
clip_model, clip_preprocess = clip.load('ViT-L/14', device=device)

# Convert to float32 — CLIP loads as float16 on GPU by default
# ArcFace overflows to NaN in float16
clip_model = clip_model.float()
clip_model.eval()

EMBED_DIM = clip_model.visual.output_dim
print(f'CLIP loaded on : {device}')
print(f'Embedding dim  : {EMBED_DIM}')  # 768 for ViT-L/14

Loading CLIP ViT-L/14...
CLIP loaded on : cuda
Embedding dim  : 768


# Build train/val & augmentation pipeline:

Define the transformations for train and val:

In [6]:
import torchvision.transforms as T
# Define separate transforms for train and val (augmentation only on train)

train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.9, 1.1),
                        interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomApply([T.ColorJitter(
        brightness=0.3, contrast=0.3, saturation=0.2, hue=0.08
    )], p=0.7),
    T.RandomGrayscale(p=0.15),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.3),
    T.ToTensor(),
    T.Normalize(mean=(0.48145466, 0.4578275,  0.40821073),
                std =(0.26862954, 0.26130258, 0.27577711)),
    T.RandomErasing(p=0.2, scale=(0.01, 0.08), ratio=(0.3, 3.3), value=0),
])

# turning val data into CLIP input
val_transform = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=(0.48145466, 0.4578275,  0.40821073),
                std =(0.26862954, 0.26130258, 0.27577711)),
])


build training and validation sets, with an 80%-20% split

In [7]:
from src.datasets import TrainDataset, SubsetWithTransform 

# original datasets
train_dataset = TrainDataset(imgs_dir, clip_preprocess, min_images=2)
NUM_CLASSES = train_dataset.num_classes

# split into 80% training 20% validations
n_train = int(0.8 * len(train_dataset))
n_val   = len(train_dataset) - n_train
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
        )

# apply the correct transforms to each subset
train_subset = SubsetWithTransform(train_subset, train_transform)
val_subset   = SubsetWithTransform(val_subset,   val_transform)

# loaders
is_cuda = (device.type == 'cuda')
train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, 
                          num_workers=2, pin_memory=is_cuda)
val_loader   = DataLoader(val_subset,   batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=is_cuda)

print(f'Train samples : {len(train_subset)}')
print(f'Val   samples : {len(val_subset)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
                                                            

Train dataset: 250 identities, 5000 images
Train samples : 4000
Val   samples : 1000
Train batches : 500
Val   batches : 63


point at the correct folders for testing:

In [8]:
# windows syntax:
# QUERY_DIR="..\\data\\competition\\test\\query"
# GALLERY_DIR="..\\data\\competition\\test\\gallery"

# linux syntax:
QUERY_DIR="../data/competition/test/query"
GALLERY_DIR="../data/competition/test/gallery"

# Zero - shot validation
Check what happens with the original weights for CLIP, before unfreezing layers and finetuning using arcface.

In [9]:
# from src.generate_submission import generate_submission
# from src.retrieval import visualise_retrieval

# # Run retrieval
# clip_model.eval()
# submission = generate_submission(
#     QUERY_DIR, GALLERY_DIR,
#     model=clip_model, preprocess=clip_preprocess, top_k=10
# )

# # Visualise 3 random queries
# visualise_retrieval(QUERY_DIR, GALLERY_DIR,
#                     clip_model, clip_preprocess,
#                     num_queries=3, top_k=10)


## ArcFace head


In [10]:
from src.arcface_head import ArcFaceHead

arcface_head = ArcFaceHead(
    embedding_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
    s=30.0,
    m=0.3
).to(device)

print(f'ArcFace head: {EMBED_DIM} → {NUM_CLASSES} classes')
print(f'Trainable params: {sum(p.numel() for p in arcface_head.parameters()):,}')

ArcFace head: 768 → 250 classes
Trainable params: 192,000


## Fine-tuning
setting up:

In [11]:
# Freeze all CLIP params
for param in clip_model.parameters():
    param.requires_grad = False

# Unfreeze last 6 transformer blocks + final LayerNorm + projection
for block in list(clip_model.visual.transformer.resblocks)[-6:]:
    for param in block.parameters():
        param.requires_grad = True
for param in clip_model.visual.ln_post.parameters():
    param.requires_grad = True
if hasattr(clip_model.visual, 'proj') and clip_model.visual.proj is not None:
    clip_model.visual.proj.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in clip_model.parameters())
print(f'Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)')

optimizer = torch.optim.AdamW([
    {'params': filter(lambda p: p.requires_grad, clip_model.parameters()), 'lr': 5e-6},
    {'params': arcface_head.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

Trainable: 76,365,824 / 427,616,513 (17.9%)


Actual finetuning:

In [12]:
torch.cuda.empty_cache()

In [ ]:
from src.train import train_one_epoch
from src.retrieval import evaluate_retrieval

EPOCHS    = 3
best_top1 = 0.0

history = {
    'loss': [],
    'top1': [],
    'top10': []
}

print('Starting fine-tuning...\n')
for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(clip_model, arcface_head, train_loader, optimizer, device, epoch)
    scheduler.step()
    top1, top10 = evaluate_retrieval(clip_model, val_loader, device)
    print(f'Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Top-1: {top1*100:.2f}% | Top-10: {top10*100:.2f}%')

    # save metrics for plotting later
    history['loss'].append(avg_loss)
    history['top1'].append(top1)
    history['top10'].append(top10)
    
    if top1 > best_top1:
        best_top1 = top1
        torch.save({
            'epoch'      : epoch,
            'clip_state' : clip_model.state_dict(),
            'head_state' : arcface_head.state_dict(),
            'top1'       : top1
        }, f'{SAVE_DIR}/best_model.pth')
        print(f'   New best saved (Top-1={top1*100:.2f}%)')

print(f'\nTraining complete. Best Top-1: {best_top1*100:.2f}%')

Starting fine-tuning...

  Epoch 1 | Batch 20/500 | Loss: 14.9873
  Epoch 1 | Batch 40/500 | Loss: 15.1895
  Epoch 1 | Batch 60/500 | Loss: 14.9085
  Epoch 1 | Batch 80/500 | Loss: 13.9539
  Epoch 1 | Batch 100/500 | Loss: 14.7034
  Epoch 1 | Batch 120/500 | Loss: 14.2663
  Epoch 1 | Batch 140/500 | Loss: 14.9060
  Epoch 1 | Batch 160/500 | Loss: 13.5385
  Epoch 1 | Batch 180/500 | Loss: 13.8387
  Epoch 1 | Batch 200/500 | Loss: 13.9132
  Epoch 1 | Batch 220/500 | Loss: 13.0736
  Epoch 1 | Batch 240/500 | Loss: 13.7933
  Epoch 1 | Batch 260/500 | Loss: 13.4613
  Epoch 1 | Batch 280/500 | Loss: 13.6544
  Epoch 1 | Batch 300/500 | Loss: 12.7843
  Epoch 1 | Batch 320/500 | Loss: 13.0555
  Epoch 1 | Batch 340/500 | Loss: 12.4560
  Epoch 1 | Batch 360/500 | Loss: 13.5039
  Epoch 1 | Batch 380/500 | Loss: 12.4656
  Epoch 1 | Batch 400/500 | Loss: 12.6093
  Epoch 1 | Batch 420/500 | Loss: 12.8378
  Epoch 1 | Batch 440/500 | Loss: 12.7403


/home/disi/miniconda3/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch 1 | Batch 460/500 | Loss: 12.4789
  Epoch 1 | Batch 480/500 | Loss: 12.9257
  Epoch 1 | Batch 500/500 | Loss: 11.7399


/home/disi/miniconda3/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Plot finetuning stats

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['loss'], marker='o')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')

ax2.plot(epochs, history['top1'], marker='o', label='Top-1')
ax2.plot(epochs, history['top10'], marker='o', label='Top-10')
ax2.set_title('Retrieval Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

## Load Best Checkpoint
⚠️ Only run this after a **session reset**.
If you just finished Cell 9 in the same session, skip this and go to Cell 11.

In [ ]:
CKPT_PATH = f'{SAVE_DIR}/best_model.pth'
ckpt = torch.load(CKPT_PATH, map_location=device)
clip_model.load_state_dict(ckpt['clip_state'])
clip_model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (Top-1 = {ckpt["top1"]*100:.2f}%)')

## Testing & visualisation

In [ ]:
from src.generate_submission import generate_submission
from src.retrieval import visualise_retrieval

# BEST MODEL PERFORMANCE
clip_model.eval()
submission = generate_submission(
    QUERY_DIR, GALLERY_DIR,
    model=clip_model, preprocess=clip_preprocess, top_k=10
)

# Visualise 3 random queriess
visualise_retrieval(QUERY_DIR, GALLERY_DIR,
                    clip_model, clip_preprocess,
                    num_queries=3, top_k=10)

## Generate Exam Submission
The evaluation website does not work anymore: this was the setup for upload and score generation

In [ ]:
# import json, requests

# def submit(results, groupname, url):
#     res = {}
#     res['groupname'] = groupname
#     res['images'] = results
#     res = json.dumps(res)
#     # print(res)
#     response = requests.post(url, res)
#     try:
#         result = json.loads(response.text)
#         print(f"accuracy is {result['accuracy']}")
#     except json.JSONDecodeError:
#         print(f"ERROR: {response.text}")

# print(submission)
# submit(results=submission,groupname="caggol",url="http://videosim.disi.unitn.it:3001/retrieval/")

# clip_model.eval()
# submission = generate_submission(
#     QUERY_DIR, GALLERY_DIR,
#     model=clip_model,
#     preprocess=clip_preprocess,
#     top_k=10
# )

# print(f'Submission ready: {len(submission)} queries')
# print(f'Example: {list(submission.items())[0]}')

# submit(submission, 'caggol')